[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/labs/lab-04-boolean-retrieval.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Lab 4 — Searching a Collection: Incidence Matrix and Inverted Index

**MSIS · AME 5053 · Week 4 · 3 hours**

In the last three labs you cleaned up single pieces of text. Today you build a tiny
**search engine**. Given a collection of documents and a Boolean query like
`cat AND NOT dog`, you will return exactly the documents that match — two ways:

1. The **term-document incidence matrix** (sessions 8), with NumPy.
2. The **inverted index** (session 9), in plain Python — the structure real search
   engines actually use.

You will build both, check they give the same answers, and see with your own numbers why
the second one is the one that scales.

**By the end of this lab you will be able to:**

1. Tokenize a collection and build its vocabulary
2. Build an incidence matrix and answer Boolean queries with bitwise operations
3. Build an inverted index of postings lists
4. Intersect two sorted postings lists with the two-pointer merge
5. Say why the inverted index wins at scale

Run every cell, in order.

---
## Part 0 — Setup

Light today — you only need `regex` (for tokenizing) and `numpy` (for the matrix). Both
are already on Colab, but the install cell runs every session because Colab wipes the
machine.

In [ ]:
!pip install -q regex numpy
print("Done.")

> **Save your own copy now:** File → Save a copy in Drive.

---
## Part 1 — The collection, and its vocabulary

A search engine works over a **collection** of documents. Here is ours: six very short
documents, numbered 1 to 6. Run it.

In [ ]:
docs = {
    1: "The cat sat on the mat.",
    2: "The dog sat on the log.",
    3: "The cat and the dog played together.",
    4: "Dogs and cats are good friends.",
    5: "A bird sang in the tall tree.",
    6: "The cat chased the bird up the tree.",
}
for doc_id, text in docs.items():
    print(doc_id, "->", text)

# Verified output:
#   1 -> The cat sat on the mat.
#   2 -> The dog sat on the log.
#   3 -> The cat and the dog played together.
#   4 -> Dogs and cats are good friends.
#   5 -> A bird sang in the tall tree.
#   6 -> The cat chased the bird up the tree.

First, turn each document into a clean set of tokens — exactly the preprocessing from lab
3: lowercase, then split into words with `regex` (never the built-in `re`). We keep the
set of *distinct* words per document, because for Boolean search all that matters is
whether a word is present, not how many times.

In [ ]:
import regex

def tokens_of(text):
    return set(regex.findall(r"\w+", text.lower()))

doc_tokens = {doc_id: tokens_of(text) for doc_id, text in docs.items()}

print("document 1 tokens:", sorted(doc_tokens[1]))
print("document 4 tokens:", sorted(doc_tokens[4]))

# Verified output:
#   document 1 tokens: ['cat', 'mat', 'on', 'sat', 'the']
#   document 4 tokens: ['and', 'are', 'cats', 'dogs', 'friends', 'good']

The **vocabulary** is every distinct term across the whole collection, sorted. This is the
list of everything the collection can be searched for.

In [ ]:
vocab = sorted(set().union(*doc_tokens.values()))
print("vocabulary size:", len(vocab))
print(vocab)

# Verified output:
#   vocabulary size: 23
#   ['a', 'and', 'are', 'bird', 'cat', 'cats', 'chased', 'dog', 'dogs', 'friends', 'good', 'in', 'log', 'mat', 'on', 'played', 'sang', 'sat', 'tall', 'the', 'together', 'tree', 'up']

Look closely: `cat` and `cats` are **both** in the vocabulary, and so are `dog` and
`dogs`. To the computer they are different words, because we did not stem them. That will
matter in a moment — remember it.

---
## Part 2 — The incidence matrix

Now build the table from session 8: one **row per term**, one **column per document**, a
**1** where the term is in the document and a **0** where it is not. We use NumPy so the
rows behave like the bit vectors from the lecture.

In [ ]:
import numpy as np

doc_ids = sorted(docs)                       # [1, 2, 3, 4, 5, 6]
term_row = {term: i for i, term in enumerate(vocab)}   # term -> row number

# rows = terms, columns = documents
A = np.zeros((len(vocab), len(doc_ids)), dtype=int)
for col, doc_id in enumerate(doc_ids):
    for term in doc_tokens[doc_id]:
        A[term_row[term], col] = 1

# show the row for 'cat' and for 'the'
print("doc:      ", doc_ids)
print("cat  row: ", A[term_row["cat"]])
print("dog  row: ", A[term_row["dog"]])
print("the  row: ", A[term_row["the"]])
print("bird row: ", A[term_row["bird"]])

# Verified output:
#   doc:       [1, 2, 3, 4, 5, 6]
#   cat  row:  [1 0 1 0 0 1]
#   dog  row:  [0 1 1 0 0 0]
#   the  row:  [1 1 1 0 1 1]
#   bird row:  [0 0 0 0 1 1]

Read the `cat` row against the document numbers: `cat` is in documents 1, 3 and 6. And
`the` is in almost every document — a sign of things to come.

### Answering a Boolean query

A Boolean query is now just bitwise arithmetic on the rows. `AND` keeps a 1 only where
both rows have one; `NOT` flips every bit. Here is a small helper and the query
`cat AND NOT dog` — "documents about cats but not dogs".

In [ ]:
import numpy as np

def matches(vector):
    # turn a row of 0/1 into the list of document IDs where it is 1
    return [doc_ids[col] for col in range(len(doc_ids)) if vector[col]]

cat = A[term_row["cat"]]
dog = A[term_row["dog"]]

result = np.logical_and(cat, 1 - dog)   # cat AND NOT dog
print("cat        ", cat)
print("dog        ", dog)
print("NOT dog    ", 1 - dog)
print("cat AND NOT dog ->", result.astype(int), "-> documents", matches(result))

# Verified output:
#   cat         [1 0 1 0 0 1]
#   dog         [0 1 1 0 0 0]
#   NOT dog     [1 0 0 1 1 1]
#   cat AND NOT dog -> [1 0 0 0 0 1] -> documents [1, 6]

Documents 1 and 6 — the cat documents, minus document 3 which also has a dog. Check it by
eye against the collection: it is right.

### Exercise 1

Answer `bird AND tree` using the matrix — documents containing **both** words. Use
`np.logical_and` and the `matches` helper.

Expected: the documents that mention a bird *and* a tree.

In [ ]:
# YOUR CODE HERE
# Pull the 'bird' and 'tree' rows, AND them, pass to matches().

---
## Part 3 — Why the matrix does not scale

Our matrix is tiny, so let us measure how much of it is wasted — how many cells are 0.

In [ ]:
import numpy as np

total_cells = A.size
ones = int(A.sum())
zeros = total_cells - ones
print(f"matrix shape : {A.shape}   (terms x documents)")
print(f"total cells  : {total_cells}")
print(f"ones (present): {ones}")
print(f"zeros        : {zeros}   ({100*zeros/total_cells:.0f}% of the matrix)")

# Verified output:
#   matrix shape : (23, 6)   (terms x documents)
#   total cells  : 138
#   ones (present): 35
#   zeros        : 103   (75% of the matrix)

Even in this toy collection most of the matrix is zeros. Now scale up, using the numbers
from session 8: a real collection might have **1,000,000 documents** and **500,000 terms**.

In [ ]:
N_docs = 1_000_000
N_terms = 500_000
cells = N_docs * N_terms
max_ones = N_docs * 1000            # each doc has ~1000 distinct words
print(f"cells in the matrix : {cells:,}")
print(f"at most this many 1s: {max_ones:,}")
print(f"fraction that are 0 : {100*(1 - max_ones/cells):.1f}%")

# Verified output:
#   cells in the matrix : 500,000,000,000
#   at most this many 1s: 1,000,000,000
#   fraction that are 0 : 99.8%

Half a **trillion** cells, and 99.8% of them are 0 — memory spent recording that words are
*absent*. That is the wall from session 8. The fix, also from session 8: stop storing the
zeros. Keep only the 1s. That is the inverted index.

---
## Part 4 — The inverted index

An inverted index maps each **term** to its **postings list**: the sorted document IDs
that contain it. No zeros — only the documents where the term actually appears.

In [ ]:
index = {}
for doc_id in doc_ids:
    for term in doc_tokens[doc_id]:
        index.setdefault(term, set()).add(doc_id)

# store each postings list SORTED — this is what makes queries fast (Part 5)
index = {term: sorted(postings) for term, postings in index.items()}

for term in ["cat", "dog", "bird", "tree", "the"]:
    print(f"{term:6} -> {index[term]}")

# Verified output:
#   cat    -> [1, 3, 6]
#   dog    -> [2, 3]
#   bird   -> [5, 6]
#   tree   -> [5, 6]
#   the    -> [1, 2, 3, 5, 6]

Compare `cat -> [1, 3, 6]` with the matrix row `1 0 1 0 0 1`: the same fact, but the
postings list stores three short numbers instead of six cells, and it never grows just
because the collection has more documents. That is the whole saving.

---
## Part 5 — Query processing: the two-pointer merge

To answer `cat AND bird` you need the documents in **both** postings lists — their
intersection. Because both lists are **sorted**, you walk them together with two pointers,
always advancing the one at the smaller ID. Each list is read once.

This is the most important function in the lab. Type it out, do not just read it.

In [ ]:
def intersect(p1, p2):
    # documents in BOTH sorted postings lists, in about len(p1)+len(p2) steps
    i = j = 0
    out = []
    while i < len(p1) and j < len(p2):
        if p1[i] == p2[j]:
            out.append(p1[i])
            i += 1
            j += 1
        elif p1[i] < p2[j]:
            i += 1          # p1[i] too small to ever match — skip it
        else:
            j += 1          # p2[j] too small — skip it
    return out

print("cat   :", index["cat"])
print("bird  :", index["bird"])
print("cat AND bird ->", intersect(index["cat"], index["bird"]))

# Verified output:
#   cat   : [1, 3, 6]
#   bird  : [5, 6]
#   cat AND bird -> [6]

`cat` is in `[1, 3, 6]`, `bird` is in `[5, 6]`, and the only document with both is **6** —
which the walk finds by stepping past 1, 3 and 5 without ever comparing every pair. On
lists of length *x* and *y* the work is about *x + y*, not *x × y*. That linear cost,
possible only because the lists are sorted, is why real search is fast.

### Handling AND NOT

`cat AND NOT dog` means "documents in `cat`'s list that are **not** in `dog`'s list". A
small variant of the same walk does it.

In [ ]:
def and_not(p1, p2):
    # documents in p1 but NOT in p2 (both sorted)
    i = j = 0
    out = []
    while i < len(p1):
        if j >= len(p2) or p1[i] < p2[j]:
            out.append(p1[i])   # p1[i] is not in p2
            i += 1
        elif p1[i] == p2[j]:
            i += 1              # in both -> exclude it
            j += 1
        else:
            j += 1
    return out

print("cat AND NOT dog ->", and_not(index["cat"], index["dog"]))

# Verified output:
#   cat AND NOT dog -> [1, 6]

`[1, 6]` — the same answer the incidence matrix gave in Part 2, reached without a single
zero stored.

### Exercise 2

Using the inverted index and your `intersect` function, answer `bird AND tree`. It should
match your Exercise 1 answer exactly — same query, different data structure, same result.

In [ ]:
# YOUR CODE HERE
# intersect the 'bird' and 'tree' postings lists.

---
## Part 6 — Same answers, and one optimisation

Prove the two methods agree. Run every single-term-pair `AND` both ways and check they
never disagree.

In [ ]:
import numpy as np

def matrix_and(t1, t2):
    return matches(np.logical_and(A[term_row[t1]], A[term_row[t2]]))

def index_and(t1, t2):
    return intersect(index[t1], index[t2])

pairs = [("cat", "bird"), ("dog", "tree"), ("cat", "dog"), ("the", "cat")]
for t1, t2 in pairs:
    m = matrix_and(t1, t2)
    x = index_and(t1, t2)
    print(f"{t1:5} AND {t2:5}: matrix={m}  index={x}  {'AGREE' if m == x else 'DIFFER!'}")

# Verified output:
#   cat   AND bird : matrix=[6]  index=[6]  AGREE
#   dog   AND tree : matrix=[]  index=[]  AGREE
#   cat   AND dog  : matrix=[3]  index=[3]  AGREE
#   the   AND cat  : matrix=[1, 3, 6]  index=[1, 3, 6]  AGREE

### The rarest-first optimisation

For `cat AND dog AND the`, intersect the **shortest** postings list first. `the` is in
almost every document, `dog` in few — starting from the small list means the fewest
documents to check.

In [ ]:
def intersect_many(terms):
    # sort the query terms by how short their postings list is: rarest first
    terms = sorted(terms, key=lambda t: len(index[t]))
    result = index[terms[0]]
    for t in terms[1:]:
        result = intersect(result, index[t])
    return result

for t in ["cat", "dog", "the"]:
    print(f"{t:5} appears in {len(index[t])} docs -> {index[t]}")
print()
print("cat AND dog AND the ->", intersect_many(["cat", "dog", "the"]))

# Verified output:
#   cat   appears in 3 docs -> [1, 3, 6]
#   dog   appears in 2 docs -> [2, 3]
#   the   appears in 5 docs -> [1, 2, 3, 5, 6]
#   
#   cat AND dog AND the -> [3]

### The normalization catch (remember Part 1)

Search for `cat`, then for `cats`. They are different terms in the index, so the search
splits documents that a human reads as the same thing.

In [ ]:
print("search 'cat'  ->", index.get("cat"))
print("search 'cats' ->", index.get("cats"))
print()
print("Document 4 ('Dogs and cats are good friends') does NOT come back for 'cat'.")

# Verified output:
#   search 'cat'  -> [1, 3, 6]
#   search 'cats' -> [4]
#   
#   Document 4 ('Dogs and cats are good friends') does NOT come back for 'cat'.

---
## Before you go

Check all of these:

- [ ] **Runtime → Restart and run all** works top to bottom with no errors
- [ ] You can build an incidence matrix and answer a query with `np.logical_and` and `NOT`
- [ ] You can build an inverted index of sorted postings lists
- [ ] You can write `intersect(p1, p2)` and explain why sorted lists make it linear
- [ ] You can say why the inverted index beats the matrix at scale
- [ ] You saved your own copy to Drive

**Take-home (optional):** add an `OR` query to the inverted index — the **union** of two
postings lists, using the same two-pointer walk but keeping every document from both. Then
add stemming to Part 1 and see which queries change.

**Next lab:** n-grams and n-gram probabilities — the start of language modelling, where we
stop searching text and start predicting it.